<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:12pt;font-weight:800;font-family:Georgia,serif;">
Gold Layer · Usine B · Génération & Enrichissement
</span><br>
<span style="font-size:9.5pt;font-family:sans-serif;color:#e8e8e8;">
Lecture depuis Silver. Génération du dataset Usine B.
Problème dominant : pannes thermiques clustérisées.
KPIs, TRS, UCL/LCL, export Parquet.
</span>
</div>

<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
00 · Imports & Configuration
</span>
</div>

In [45]:
# ============================================================
# GOLD LAYER — USINE B
# Input  : data/silver/ai4i_clean.parquet
# Output : data/gold/usine_B.parquet
#          data/gold/limites_usine_B.json
# ============================================================

import pandas as pd
import numpy as np
import json
import logging
from pathlib import Path
from datetime import datetime
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger('gold_B')

ROOT        = Path('..').resolve()
SILVER_FILE = ROOT / 'data' / 'silver' / 'ai4i_clean.parquet'
GOLD_PATH   = ROOT / 'data' / 'gold'
GOLD_PATH.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

np.random.seed(123)

LAYOUT = dict(
    paper_bgcolor='#ffffff',
    plot_bgcolor='#ffffff',
    font=dict(family='Inter, sans-serif', color='#374151', size=11),
    hoverlabel=dict(bgcolor='#1f2937', font_color='#f9fafb', font_size=11),
)

# ── Paramètres physiques Usine B ─────────────────────────────
# Moteur plus puissant — pièces REF02 nécessitent plus de couple
# N = (P × 60) / (2π × T)
PUISSANCE_W      = 7_200.0
COUPLE_NOMINAL   = 45.0
VITESSE_NOMINALE = round((PUISSANCE_W * 60) / (2 * np.pi * COUPLE_NOMINAL))
TEMPS_CYCLE_NOM  = round(60_000 / VITESSE_NOMINALE, 2)
DUREE_VIE_OUTIL  = 250

TEMP_AIR_BASE    = 23.0
OFFSET_THERMIQUE = 11.5

# Seuil critique HDF mesuré sur le diagnostic Bronze (Plot 6)
SEUIL_DELTA_HDF  = 8.6

# Paramètres du processus de Poisson non homogène
# λ_base = taux HDF mesuré sur AI4I brut
# T_cluster = 45 jours × 48 sessions/jour
LAMBDA_BASE      = 0.0115
AMPLITUDE        = 0.85
T_CLUSTER        = 45 * 48

REBUT_BASE       = 0.03
USINE_ID         = 'Usine_B'
MACHINES         = ['M01', 'M02', 'M03', 'M04']
TYPES_PIECES     = ['REF01', 'REF02', 'REF03', 'REF04']
POIDS_PIECES     = [0.35, 0.35, 0.20, 0.10]
N_SESSIONS       = 6 * 30 * 24 * 2
START_DATE       = '2024-01-01 06:00'

log.info("Configuration Gold Usine B chargée")
print(f"\n  Vitesse nominale   : {VITESSE_NOMINALE} tr/min  ← N = P×60/(2π×T)")
print(f"  Temps cycle nominal: {TEMPS_CYCLE_NOM} min      ← t = 60 000/N")
print(f"  Seuil critique HDF : {SEUIL_DELTA_HDF} K        ← mesuré Bronze Plot 6")
print(f"  Lambda base pannes : {LAMBDA_BASE}              ← taux HDF AI4I brut")
print(f"  Période clusters   : {T_CLUSTER} sessions        ← 45 jours × 48 sessions/j")

14:45:46 | INFO | Configuration Gold Usine B chargée



  Vitesse nominale   : 1528 tr/min  ← N = P×60/(2π×T)
  Temps cycle nominal: 39.27 min      ← t = 60 000/N
  Seuil critique HDF : 8.6 K        ← mesuré Bronze Plot 6
  Lambda base pannes : 0.0115              ← taux HDF AI4I brut
  Période clusters   : 2160 sessions        ← 45 jours × 48 sessions/j


<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
01 · Lecture Silver & structure temporelle
</span>
</div>

In [46]:
# ============================================================
# LECTURE SILVER & STRUCTURE TEMPORELLE USINE B
# ============================================================

df_silver = pd.read_parquet(SILVER_FILE)
log.info("Silver chargé : %d lignes · %d colonnes", *df_silver.shape)

timestamps   = pd.date_range(start=START_DATE, periods=N_SESSIONS, freq='30min')
df           = pd.DataFrame({'timestamp': timestamps})
df['usine_id']   = USINE_ID
df['machine_id'] = np.random.choice(MACHINES, size=N_SESSIONS)
df['type_piece'] = np.random.choice(TYPES_PIECES, size=N_SESSIONS, p=POIDS_PIECES)

def assigner_equipe(ts):
    h = ts.hour
    if 6 <= h < 14:    return 'Matin'
    elif 14 <= h < 22: return 'Apres-midi'
    else:               return 'Nuit'

df['equipe'] = df['timestamp'].apply(assigner_equipe)

print(f"\n  Sessions générées : {len(df):,}")
print(f"  Période           : {df['timestamp'].min()} → {df['timestamp'].max()}")
print(f"  Machines          : {df['machine_id'].value_counts().to_dict()}")
print(f"  Pièces            : {df['type_piece'].value_counts().to_dict()}")
print()
display(df.head(5))

14:45:46 | INFO | Silver chargé : 9985 lignes · 15 colonnes



  Sessions générées : 8,640
  Période           : 2024-01-01 06:00:00 → 2024-06-29 05:30:00
  Machines          : {'M03': 2236, 'M04': 2172, 'M01': 2118, 'M02': 2114}
  Pièces            : {'REF01': 3050, 'REF02': 3042, 'REF03': 1742, 'REF04': 806}



,timestamp,usine_id,machine_id,type_piece,equipe
0,2024-01-01 06:00:00,Usine_B,M03,REF03,Matin
1,2024-01-01 06:30:00,Usine_B,M02,REF01,Matin
2,2024-01-01 07:00:00,Usine_B,M03,REF02,Matin
3,2024-01-01 07:30:00,Usine_B,M03,REF02,Matin
4,2024-01-01 08:00:00,Usine_B,M01,REF04,Matin


<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
02 · Génération des capteurs — signal thermique dominant
</span>
</div>

In [47]:
# ============================================================
# CAPTEURS USINE B — Signal thermique dominant
#
# Le delta thermique varie de façon sinusoïdale :
# delta(t) = offset_nominal - D×sin(2π×t/T_cluster) + bruit
#
# Quand sin → 1 : delta descend → zone critique HDF
# Quand sin → -1 : delta monte → refroidissement efficace
# D = 2.5 K : amplitude calibrée pour déclencher HDF
# ============================================================

t = np.arange(N_SESSIONS)
D = 2.5

df['temp_air_c'] = np.round(
    np.random.normal(TEMP_AIR_BASE, 1.5, N_SESSIONS), 2)

delta_thermique = (
    OFFSET_THERMIQUE
    - D * np.sin(2 * np.pi * t / T_CLUSTER)
    + np.random.normal(0, 0.5, N_SESSIONS)
)

df['temp_process_delta'] = np.round(delta_thermique, 2)
df['temp_process_c']     = np.round(df['temp_air_c'] + df['temp_process_delta'], 2)

df['vitesse_rotation_rpm'] = np.clip(
    np.random.normal(VITESSE_NOMINALE, 80, N_SESSIONS).astype(int), 1000, 2000)
df['couple_nm'] = np.round(
    np.clip(np.random.normal(COUPLE_NOMINAL, 7, N_SESSIONS), 25, 70), 2)

# Usure outil — remplacement à 85% (plus rigoureux que Usine A)
increments = {'REF01':(2.0,3.0),'REF02':(2.8,4.0),
              'REF03':(3.5,5.0),'REF04':(2.0,4.0)}

usure_actuelle = 0.0
usure_list, remplacement_list = [], []

for i in range(N_SESSIONS):
    lo, hi = increments[df.loc[i,'type_piece']]
    usure_actuelle += np.random.uniform(lo, hi)
    if usure_actuelle >= DUREE_VIE_OUTIL * 0.85:
        remplacement_list.append(1)
        usure_actuelle = 0.0
    else:
        remplacement_list.append(0)
    usure_list.append(round(usure_actuelle, 1))

df['usure_outil_min']    = usure_list
df['remplacement_outil'] = remplacement_list
df['usure_cumulee_pct']  = np.round(
    (df['usure_outil_min'] / DUREE_VIE_OUTIL * 100).clip(0,100), 2)

print(f"  Delta thermique moy : {df['temp_process_delta'].mean():.2f} K")
print(f"  Delta thermique min : {df['temp_process_delta'].min():.2f} K  ← zone critique")
print(f"  Delta thermique max : {df['temp_process_delta'].max():.2f} K")
print(f"  Remplacements outil : {sum(remplacement_list)}")

  Delta thermique moy : 11.49 K
  Delta thermique min : 7.47 K  ← zone critique
  Delta thermique max : 15.55 K
  Remplacements outil : 129


<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
03 · Pannes thermiques clustérisées — Poisson non homogène
</span>
</div>

In [48]:
# ============================================================
# PANNES USINE B — Processus de Poisson non homogène
#
# λ(t) = λ_base × (1 + A × sin(2π×t/T_cluster))
# Facteur thermique × 3 si delta_T < seuil_HDF
#
# p_HDF(t) = λ(t) × facteur_thermique, borné à 0.5
# ============================================================

lambda_t = LAMBDA_BASE * (1 + AMPLITUDE * np.sin(2 * np.pi * t / T_CLUSTER))

facteur_thermique  = np.where(df['temp_process_delta'] < SEUIL_DELTA_HDF, 3.0, 1.0)
prob_panne_thermique = (lambda_t * facteur_thermique).clip(0, 0.5)

panne_thermique = (np.random.random(N_SESSIONS) < prob_panne_thermique).astype(int)

prob_usure  = (df['usure_cumulee_pct'] / 100) ** 2 * 0.08
panne_usure = ((np.random.random(N_SESSIONS) < prob_usure) &
               (df['usure_cumulee_pct'] > 70)).astype(int)

panne_surge = (np.random.random(N_SESSIONS) < 0.003).astype(int)
panne_alea  = (np.random.random(N_SESSIONS) < 0.001).astype(int)

panne_machine = ((panne_thermique==1)|(panne_usure==1)|
                 (panne_surge==1)|(panne_alea==1)).astype(int)

df['panne_machine']     = panne_machine
df['panne_usure_outil'] = panne_usure
df['panne_thermique']   = panne_thermique
df['panne_puissance']   = 0
df['panne_surcharge']   = panne_surge
df['panne_aleatoire']   = panne_alea

def type_panne(row):
    if row['panne_thermique']:   return 'Thermique'
    if row['panne_usure_outil']: return 'Usure outil'
    if row['panne_surcharge']:   return 'Surcharge'
    if row['panne_aleatoire']:   return 'Aleatoire'
    return 'Aucune'

df['type_panne'] = df.apply(type_panne, axis=1)

durees = []
for _, row in df.iterrows():
    if row['panne_thermique']:   durees.append(round(np.random.uniform(90,240),1))
    elif row['panne_usure_outil']: durees.append(round(np.random.uniform(60,180),1))
    elif row['panne_surcharge']: durees.append(round(np.random.uniform(30,90),1))
    elif row['panne_aleatoire']: durees.append(round(np.random.uniform(15,45),1))
    else: durees.append(0.0)

df['duree_arret_min']  = durees
df['nb_pannes_7j']     = df['panne_machine'].rolling(336, min_periods=1).sum().astype(int)
df['intervention_maintenance'] = (
    (df['panne_machine']==1)|(df['nb_pannes_7j']>5)).astype(int)

print(f"  Taux panne global    : {df['panne_machine'].mean()*100:.2f}%")
print(f"  Pannes thermiques    : {df['panne_thermique'].sum()}")
print(f"  Durée arrêt moy      : {df[df['duree_arret_min']>0]['duree_arret_min'].mean():.1f} min")
print(f"  Interventions maint. : {df['intervention_maintenance'].sum()}")

  Taux panne global    : 2.55%
  Pannes thermiques    : 111
  Durée arrêt moy      : 135.1 min
  Interventions maint. : 7058


<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
04 · KPIs production
</span>
</div>

In [49]:
# ============================================================
# KPIs USINE B
# Rebut piloté par le delta thermique :
# τ = τ_base + α×max(0, seuil_HDF - delta_T) + β×panne + bruit
# α = 0.05 : chaque Kelvin sous le seuil ajoute 5% de rebut
# ============================================================

df['temps_cycle_min'] = np.round(
    (TEMPS_CYCLE_NOM * (1 + 0.001 * df['usure_cumulee_pct'])
     + np.random.normal(0, 1.8, N_SESSIONS)).clip(lower=20), 2)

df['derive_cycle_pct'] = np.round(
    (df['temps_cycle_min'] - TEMPS_CYCLE_NOM) / TEMPS_CYCLE_NOM * 100, 2)

df['pieces_produites'] = (30 / df['temps_cycle_min']).astype(int).clip(lower=1)
df.loc[df['panne_machine']==1, 'pieces_produites'] = 0

df['taux_rebut'] = np.round(
    (REBUT_BASE
     + 0.05 * np.maximum(0, SEUIL_DELTA_HDF - df['temp_process_delta'])
     + 0.12 * df['panne_machine']
     + np.random.normal(0, 0.005, N_SESSIONS)).clip(0,1), 4)

df['pieces_rebutees'] = (df['pieces_produites'] * df['taux_rebut']).astype(int)

df['indice_stabilite_process'] = np.round(
    (0.90
     - 0.05 * np.maximum(0, SEUIL_DELTA_HDF - df['temp_process_delta'])
     - 0.25 * df['panne_machine']
     + np.random.normal(0, 0.03, N_SESSIONS)).clip(0,1), 4)

df['consommation_kwh'] = np.round(
    ((df['vitesse_rotation_rpm'] * df['couple_nm']) / 9550 * 0.5
     + np.random.normal(0, 0.5, N_SESSIONS)).clip(lower=0), 2)

mask_appro = (df['timestamp'] >= '2024-04-20') & (df['timestamp'] <= '2024-04-25')
df['rupture_appro'] = mask_appro.astype(int)
df.loc[mask_appro, 'pieces_produites'] = 0

print(f"  Temps cycle moyen   : {df['temps_cycle_min'].mean():.2f} min")
print(f"  Taux rebut moyen    : {df['taux_rebut'].mean()*100:.2f}%")
print(f"  Taux rebut max      : {df['taux_rebut'].max()*100:.2f}%")
print(f"  Pièces produites    : {df['pieces_produites'].sum():,}")
print(f"  Conso énergie moy   : {df['consommation_kwh'].mean():.2f} kWh")

  Temps cycle moyen   : 40.92 min
  Taux rebut moyen    : 3.34%
  Taux rebut max      : 18.25%
  Pièces produites    : 8,185
  Conso énergie moy   : 3.61 kWh


<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
05 · TRS & Limites de contrôle SPC
</span>
</div>

In [50]:
# ============================================================
# TRS & SPC — USINE B
# ============================================================

df['disponibilite'] = (df['panne_machine']==0).astype(float)
df['performance']   = np.round((df['vitesse_rotation_rpm']/VITESSE_NOMINALE).clip(upper=1.0), 4)
df['qualite']       = np.round(1 - df['taux_rebut'], 4)
df['TRS']           = np.round(df['disponibilite']*df['performance']*df['qualite'], 4)

baseline = df[(df['timestamp'] < '2024-02-01') & (df['panne_machine']==0)]

def calcul_limites(serie, nom, n_sigma=3):
    mu, sigma = serie.mean(), serie.std()
    return {
        'kpi'    : nom,
        'moyenne': round(mu, 4),
        'sigma'  : round(sigma, 4),
        'ucl'    : round(mu + n_sigma*sigma, 4),
        'lcl'    : round(max(mu - n_sigma*sigma, 0), 4),
        'uwl'    : round(mu + 2*sigma, 4),
        'lwl'    : round(max(mu - 2*sigma, 0), 4),
    }

kpis = {
    'TRS'                : baseline['TRS'],
    'taux_rebut'         : baseline['taux_rebut'],
    'temps_cycle_min'    : baseline['temps_cycle_min'],
    'temp_process_delta' : baseline['temp_process_delta'],
    'consommation_kwh'   : baseline['consommation_kwh'],
    'indice_stabilite'   : baseline['indice_stabilite_process'],
}

limites = {nom: calcul_limites(serie, nom) for nom, serie in kpis.items()}
df_lim  = pd.DataFrame(limites.values()).set_index('kpi')

print(f"  TRS moyen global : {df['TRS'].mean()*100:.1f}%")
print(f"  TRS moyen S1     : {df[df['timestamp']<'2024-02-01']['TRS'].mean()*100:.1f}%")
print(f"  TRS moyen S6     : {df[df['timestamp']>='2024-06-01']['TRS'].mean()*100:.1f}%")
print()
display(df_lim)

  TRS moyen global : 92.6%
  TRS moyen S1     : 92.3%
  TRS moyen S6     : 93.0%



,moyenne,sigma,ucl,lcl,uwl,lwl
kpi,,,,,,
TRS,0.95,0.03,1.04,0.86,1.01,0.89
taux_rebut,0.03,0.01,0.05,0.01,0.04,0.02
temps_cycle_min,40.95,2.05,47.10,34.80,45.05,36.85
temp_process_delta,10.68,1.58,15.43,5.93,13.84,7.51
consommation_kwh,3.62,0.77,5.94,1.30,5.17,2.07
indice_stabilite,0.90,0.03,0.99,0.81,0.96,0.84


<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
06 · Visualisations Gold — dérives Usine B
</span>
</div>

In [51]:
# ── Plot 1 : Delta thermique et clusters de pannes ───────────
sample = df.iloc[::3].copy()
df['delta_ma'] = df['temp_process_delta'].rolling(100, min_periods=1).mean()

fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=sample['timestamp'], y=sample['temp_process_delta'],
    name='Delta thermique (K)', line=dict(color='#c7d2fe', width=0.8), opacity=0.6,
    hovertemplate='%{x|%d %b %H:%M}<br>Delta T : %{y:.2f} K<extra></extra>'
))
fig1.add_trace(go.Scatter(
    x=df['timestamp'], y=df['delta_ma'],
    name='Moyenne mobile 100 sessions', line=dict(color='#6366f1', width=2.2),
    hovertemplate='%{x|%d %b}<br>Moy Delta T : %{y:.2f} K<extra></extra>'
))

mask_hdf = df['panne_thermique'] == 1
fig1.add_trace(go.Scatter(
    x=df.loc[mask_hdf, 'timestamp'],
    y=df.loc[mask_hdf, 'temp_process_delta'],
    mode='markers', name='Panne thermique',
    marker=dict(color='#ef4444', size=5, symbol='x',
                line=dict(width=1.5, color='#ef4444')),
    hovertemplate='%{x|%d %b %H:%M}<br>Delta T : %{y:.2f} K<br><b>HDF</b><extra></extra>'
))

fig1.add_hline(y=SEUIL_DELTA_HDF, line_dash='dot', line_color='#ef4444', line_width=1.2,
               annotation_text=f'Seuil critique HDF = {SEUIL_DELTA_HDF} K',
               annotation_font_size=9, annotation_font_color='#ef4444',
               annotation_position='top right')

fig1.update_layout(
    **LAYOUT,
    title=dict(text='Usine B : Delta thermique et clusters de pannes HDF',
               font=dict(size=12, color='#111827'), x=0.01),
    xaxis=dict(gridcolor='#f3f4f6', linecolor='#e5e7eb',
               tickfont=dict(size=10, color='#9ca3af'), tickformat='%d %b'),
    yaxis=dict(title='Delta T (K)', gridcolor='#f3f4f6', linecolor='#e5e7eb',
               tickfont=dict(size=10, color='#9ca3af')),
    legend=dict(bgcolor='rgba(0,0,0,0)', borderwidth=0,
                font=dict(size=10), orientation='h', y=1.08),
    height=400, margin=dict(l=50,r=20,t=60,b=40),
    hovermode='x unified'
)
fig1.show()

In [52]:
# ── Plot 2 : TRS Usine B — comportement par vagues ───────────
df['TRS_7j'] = df['TRS'].rolling(336, min_periods=1).mean()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=sample['timestamp'], y=sample['TRS'],
    name='TRS session', line=dict(color='#fed7aa', width=0.8), opacity=0.6,
    hovertemplate='%{x|%d %b %H:%M}<br>TRS : %{y:.3f}<extra></extra>'
))
fig2.add_trace(go.Scatter(
    x=df['timestamp'], y=df['TRS_7j'],
    name='Moyenne mobile 7j', line=dict(color='#f97316', width=2.5),
    hovertemplate='%{x|%d %b}<br>TRS moy 7j : %{y:.3f}<extra></extra>'
))
fig2.add_hline(y=limites['TRS']['lcl'], line_dash='dot', line_color='#ef4444',
               line_width=1.2,
               annotation_text=f"LCL = {limites['TRS']['lcl']:.3f}",
               annotation_font_size=9, annotation_font_color='#ef4444',
               annotation_position='bottom right')

fig2.update_layout(
    **LAYOUT,
    title=dict(text='Usine B : TRS sur 6 mois (comportement par vagues thermiques)',
               font=dict(size=12, color='#111827'), x=0.01),
    xaxis=dict(gridcolor='#f3f4f6', linecolor='#e5e7eb',
               tickfont=dict(size=10, color='#9ca3af'), tickformat='%d %b'),
    yaxis=dict(title='TRS', gridcolor='#f3f4f6', linecolor='#e5e7eb',
               tickfont=dict(size=10, color='#9ca3af'), range=[0,1.05]),
    legend=dict(bgcolor='rgba(0,0,0,0)', borderwidth=0,
                font=dict(size=10), orientation='h', y=1.08),
    height=380, margin=dict(l=50,r=20,t=60,b=40),
    hovermode='x unified'
)
fig2.show()

<div style="border-left:4px solid #ffffff;padding:10px 18px;margin-bottom:2px;">
<span style="font-size:10pt;font-weight:700;font-family:Georgia,serif;">
07 · Export Gold
</span>
</div>

In [53]:
# ============================================================
# EXPORT GOLD — USINE B
# ============================================================

def json_serial(obj):
    if isinstance(obj, np.integer): return int(obj)
    if isinstance(obj, np.floating): return float(obj)
    if isinstance(obj, np.ndarray):  return obj.tolist()
    raise TypeError(f"Type {type(obj)} non sérialisable")

gold_file = GOLD_PATH / 'usine_B.parquet'
lim_file  = GOLD_PATH / 'limites_usine_B.json'

df.to_parquet(gold_file, index=False)

with open(lim_file, 'w') as f:
    json.dump(limites, f, indent=2, default=json_serial)

log.info("Gold Usine B exporté : %s", gold_file)

print(f"\n{'='*55}")
print(f"  GOLD OUTPUTS — USINE B")
print(f"{'='*55}")
print(f"  ✓ {gold_file}")
print(f"  ✓ {lim_file}")
print(f"  Shape          : {df.shape[0]:,} lignes · {df.shape[1]} colonnes")
print(f"  TRS moyen      : {df['TRS'].mean()*100:.1f}%")
print(f"  Taux rebut moy : {df['taux_rebut'].mean()*100:.2f}%")
print(f"  Pannes HDF     : {df['panne_thermique'].sum()}")
print(f"{'='*55}")

14:45:48 | INFO | Gold Usine B exporté : C:\Users\j.innouri\OneDrive - pellencst.com\Bureau\MSPR_TPRE831\data\gold\usine_B.parquet



  GOLD OUTPUTS — USINE B
  ✓ C:\Users\j.innouri\OneDrive - pellencst.com\Bureau\MSPR_TPRE831\data\gold\usine_B.parquet
  ✓ C:\Users\j.innouri\OneDrive - pellencst.com\Bureau\MSPR_TPRE831\data\gold\limites_usine_B.json
  Shape          : 8,640 lignes · 37 colonnes
  TRS moyen      : 92.6%
  Taux rebut moy : 3.34%
  Pannes HDF     : 111
